In [1]:
%load_ext autoreload
%autoreload 2
import warnings

import os
import time
import pyproj
import numpy as np
import xarray as xr
import rioxarray
import s1_ard_core.computing.computation
import s1_ard_core.computing.coreg
import s1_ard_core.computing.dem
import s1_ard_core.computing.etad
import s1_ard_core.computing.gslc
import s1_ard_core.computing.io
import s1_ard_core.computing.orbit
import s1_ard_core.computing.shiroma
import s1_ard_core.computing.simul
from s1_ard_core.computing.io import read, write, normpath,Path
import s1_ard_core.computing.slc

import cop_ard_core.esd as esd_module
from s1_ard_core.computing.logger import logger
from pyproj.transformer import TransformerGroup

os.environ["PROJ_DATA"] = pyproj.datadir.get_data_dir()
warnings.filterwarnings("ignore")

In [2]:
type, n_jobs, job_memory = "LOCAL", 12, "16GB"
client = s1_ard_core.computing.computation.create_client(type, n_jobs, job_memory)

2026-01-23 12:19:48 [info     ] <Cluster info: type=LOCAL, n_jobs=12, job_memory=16GB>


In [3]:
kml_path = "/home/morea/SENTINEL1/KML/S2A_OPER_GIP_TILPAR_MPC__20151209T095117_V20150622T000000_21000101T000000_B00.kml"
in_files = [
    "/home/morea/SENTINEL1/SALZBURG/44/SAFE/S1A_IW_SLC__1SDV_20241015T165929_20241015T165956_056116_06DDD0_2FE3.SAFE",
    "/home/morea/SENTINEL1/SALZBURG/44/SAFE/S1A_IW_SLC__1SDV_20241027T165929_20241027T165956_056291_06E4C5_095D.SAFE",
]
etad_files = None

path = "/home/morea/SENTINEL1/SALZBURG/44/S1-RP/ESD"
slcs_path = f"{path}/slcs"
wdir_path_static = f"{path}/STATIC"
wdir_path_tmp = f"{path}/DYNAMIC"

orbit_path = None
reference_date = "2024-10-15"
chunks_sr = {"acquisition_date": 1, "polarization": 1, "azimuth_time": 2048, "slant_range_time": 2048}
chunks = {"x": 2048, "y": 2048}
chunk_length = max([v for v in chunks.values()])
spacing = {"x": 30, "y": 30}
coreg = True
gamma_flattening = True
crs = None
recompute = False
multiburst = True

In [4]:
slcs = read(slcs_path, consolidate_metadata=False)

In [5]:
dem_tree = xr.DataTree()
for k, v in (slcs.bursts if multiburst else {"data": slcs}).items():
    path = f"bursts/{k}" if multiburst else "data"
    d = s1_ard_core.computing.io.read(f"{wdir_path_tmp}/{path}/dem_resampled", consolidate_metadata=False)
    if multiburst:
        dem_tree[path] = d
    else:
        dem_tree = xr.DataTree(d)

In [6]:
orbits = s1_ard_core.computing.slc.extract_orbits(
    slcs,
    multiburst=multiburst,
    acquisition_date=[reference_date],
)
simulation_tree = s1_ard_core.computing.simul.stack_simulation(
    dem_tree=dem_tree,
    orbits=orbits,
    multiburst=multiburst,
    working_directory=wdir_path_static,
    chunks={**chunks, "acquisition_date": 1},
    master_image=True,
    recompute=recompute,
)
simulation_ml_tree = s1_ard_core.computing.simul.stack_simulation_ml(
    simulation_tree=simulation_tree,
    reference_date=reference_date,
    multiburst=multiburst,
    working_directory=wdir_path_static,
    recompute=recompute,
)
simulation_secondary_tree = xr.DataTree()
for k in simulation_tree.bursts if multiburst else {"data": simulation_tree}:
    path = f"bursts/{k}" if multiburst else "data"
    data_path = s1_ard_core.computing.io.normpath(f"{wdir_path_tmp}/{path}/simulation")
    if multiburst:
        simulation_secondary_tree[path] = s1_ard_core.computing.io.read(data_path, consolidate_metadata=False)
    else:
        simulation_secondary_tree = s1_ard_core.computing.io.read(data_path, consolidate_metadata=False)

2026-01-23 12:19:55 [info     ] Found existing master image simulation file for iw2_92640
2026-01-23 12:19:55 [info     ] Found existing master image simulation file for iw1_92638
2026-01-23 12:19:55 [info     ] Found existing master image simulation file for iw1_92645
2026-01-23 12:19:55 [info     ] Found existing master image simulation file for iw1_926442026-01-23 12:19:55 [info     ] Found existing master image simulation file for iw1_92641

2026-01-23 12:19:55 [info     ] Found existing master image simulation file for iw1_92643
2026-01-23 12:19:56 [info     ] Found existing master image simulation file for iw1_92639
2026-01-23 12:19:56 [info     ] Found existing master image simulation file for iw1_92646
2026-01-23 12:19:56 [info     ] Found existing master image simulation file for iw2_92638
2026-01-23 12:19:56 [info     ] Found existing master image simulation file for iw3_92642
2026-01-23 12:19:56 [info     ] Found existing master image simulation file for iw3_92638
2026-01-23

In [7]:
files = s1_ard_core.computing.io.glob(f"{wdir_path_tmp}/bursts/iw*")
for f in files:
    k = f.split("/")[-1]
    slcs[f"bursts/{k}/coreg"] = s1_ard_core.computing.io.read(f"{f}/coreg", consolidate_metadata=True)

In [8]:
y_poly = esd_module.estimate(slcs, wdir_path_tmp, reference_date, poly_degree=2, enable_dask=True)
x_poly = None

/home/morea/Desktop/ESA/rosel-management/cop-ard-core/.venv/lib/python3.11/site-packages/dask/dataframe/__init__.py:31: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)
/home/morea/Desktop/ESA/rosel-management/cop-ard-core/.venv/lib/python3.11/site-packages/dask/dataframe/__init__.py:31: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)
/home/morea/Desktop/ESA/rosel-management/cop-ard-core/.venv/lib/python3.11/site-packages/dask/dataframe/__init__.py:31: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[datafra

2026-01-23 12:20:09 [info     ] Processing overlap area between bursts: iw1_92638 and iw1_92639
2026-01-23 12:20:09 [info     ] Processing overlap area between bursts: iw1_92642 and iw1_92643
2026-01-23 12:20:09 [info     ] Processing overlap area between bursts: iw1_92643 and iw1_92644
2026-01-23 12:20:09 [info     ] Processing overlap area between bursts: iw2_92639 and iw2_92640
2026-01-23 12:20:09 [info     ] Processing overlap area between bursts: iw1_92641 and iw1_92642
2026-01-23 12:20:10 [info     ] Processing overlap area between bursts: iw1_92644 and iw1_92645
2026-01-23 12:20:10 [info     ] Processing overlap area between bursts: iw1_92640 and iw1_92641
2026-01-23 12:20:10 [info     ] ESD computation skipped: burst (iw1_92647) does not exist
2026-01-23 12:20:10 [info     ] Processing overlap area between bursts: iw1_92645 and iw1_92646
2026-01-23 12:20:10 [info     ] Processing overlap area between bursts: iw3_92639 and iw3_92640
2026-01-23 12:20:10 [info     ] Processing ove

/home/morea/Desktop/ESA/rosel-management/cop-ard-core/.venv/lib/python3.11/site-packages/dask/base.py:1462: UserWarning: Running on a single-machine scheduler when a distributed client is active might lead to unexpected results.
  warnings.warn(
/home/morea/Desktop/ESA/rosel-management/cop-ard-core/.venv/lib/python3.11/site-packages/dask/base.py:1462: UserWarning: Running on a single-machine scheduler when a distributed client is active might lead to unexpected results.
  warnings.warn(
/home/morea/Desktop/ESA/rosel-management/cop-ard-core/.venv/lib/python3.11/site-packages/dask/base.py:1462: UserWarning: Running on a single-machine scheduler when a distributed client is active might lead to unexpected results.
  warnings.warn(
/home/morea/Desktop/ESA/rosel-management/cop-ard-core/.venv/lib/python3.11/site-packages/dask/base.py:1462: UserWarning: Running on a single-machine scheduler when a distributed client is active might lead to unexpected results.
  warnings.warn(
/home/morea/Desk

2026-01-23 12:20:10 [info     ] Processing overlap area between bursts: iw2_92643 and iw2_92644
2026-01-23 12:20:10 [info     ] Computing of azimuth shifts   
2026-01-23 12:20:10 [info     ] Processing overlap area between bursts: iw3_92638 and iw3_92639
2026-01-23 12:20:10 [info     ] Computing of azimuth shifts   
2026-01-23 12:20:10 [info     ] Processing overlap area between bursts: iw2_92642 and iw2_92643
2026-01-23 12:20:10 [info     ] Computing of azimuth shifts   
2026-01-23 12:20:10 [info     ] Processing overlap area between bursts: iw3_92643 and iw3_92644
2026-01-23 12:20:10 [info     ] Processing overlap area between bursts: iw3_92642 and iw3_92643
2026-01-23 12:20:10 [info     ] Processing overlap area between bursts: iw3_92640 and iw3_92641
2026-01-23 12:20:10 [info     ] Computing of azimuth shifts   
2026-01-23 12:20:10 [info     ] Computing of azimuth shifts   
2026-01-23 12:20:10 [info     ] Computing of azimuth shifts   
2026-01-23 12:20:10 [info     ] ESD computatio

/home/morea/Desktop/ESA/rosel-management/cop-ard-core/.venv/lib/python3.11/site-packages/dask/base.py:1462: UserWarning: Running on a single-machine scheduler when a distributed client is active might lead to unexpected results.
  warnings.warn(
/home/morea/Desktop/ESA/rosel-management/cop-ard-core/.venv/lib/python3.11/site-packages/dask/base.py:1462: UserWarning: Running on a single-machine scheduler when a distributed client is active might lead to unexpected results.
  warnings.warn(
/home/morea/Desktop/ESA/rosel-management/cop-ard-core/.venv/lib/python3.11/site-packages/dask/base.py:1462: UserWarning: Running on a single-machine scheduler when a distributed client is active might lead to unexpected results.
  warnings.warn(
/home/morea/Desktop/ESA/rosel-management/cop-ard-core/.venv/lib/python3.11/site-packages/dask/base.py:1462: UserWarning: Running on a single-machine scheduler when a distributed client is active might lead to unexpected results.
  warnings.warn(
/home/morea/Desk


2026-01-23 12:20:10 [info     ] Number of blocks to be processed: 2
2026-01-23 12:20:10 [info     ] Selected scatter strategy: once-and-for-all
2026-01-23 12:20:10 [info     ] Number of blocks to be processed: 2


/home/morea/Desktop/ESA/rosel-management/cop-ard-core/.venv/lib/python3.11/site-packages/dask/base.py:1462: UserWarning: Running on a single-machine scheduler when a distributed client is active might lead to unexpected results.
  warnings.warn(
/home/morea/Desktop/ESA/rosel-management/cop-ard-core/.venv/lib/python3.11/site-packages/dask/base.py:1462: UserWarning: Running on a single-machine scheduler when a distributed client is active might lead to unexpected results.
  warnings.warn(
/home/morea/Desktop/ESA/rosel-management/cop-ard-core/.venv/lib/python3.11/site-packages/dask/base.py:1462: UserWarning: Running on a single-machine scheduler when a distributed client is active might lead to unexpected results.
  warnings.warn(
/home/morea/Desktop/ESA/rosel-management/cop-ard-core/.venv/lib/python3.11/site-packages/dask/base.py:1462: UserWarning: Running on a single-machine scheduler when a distributed client is active might lead to unexpected results.
  warnings.warn(
/home/morea/Desk

2026-01-23 12:20:53 [debug    ] A : (24, 6)                   
2026-01-23 12:20:53 [debug    ] B : (24,)                     


In [9]:
slcs = s1_ard_core.computing.coreg.stack_coregistration(
    slcs,
    simulation_tree,
    simulation_ml_tree,
    reference_date,
    simulation_secondary_tree=simulation_secondary_tree,
    multiburst=multiburst,
    working_directory=f"{wdir_path_tmp}_ESD",
    y_poly=y_poly,
    recompute=True,
)

2026-01-22 21:55:10 [info     ] Selected scatter strategy: once-and-for-all
2026-01-22 21:55:10 [warning  ] The chunksize for dimension acquisition_date is larger than data size along with this dimension. In such a case for this dimension a single chunk of size 10 will be used
2026-01-22 21:55:10 [warning  ] The chunksize for dimension azimuth_time is larger than data size along with this dimension. In such a case for this dimension a single chunk of size 1502 will be used
2026-01-22 21:55:10 [info     ] Number of blocks to be processed: 12
2026-01-22 21:55:12 [info     ] Selected scatter strategy: once-and-for-all
2026-01-22 21:55:12 [warning  ] The chunksize for dimension acquisition_date is larger than data size along with this dimension. In such a case for this dimension a single chunk of size 10 will be used
2026-01-22 21:55:12 [warning  ] The chunksize for dimension azimuth_time is larger than data size along with this dimension. In such a case for this dimension a single chunk o